# CSV1 被接管用户价格补全核验

逐行核验 22,044 笔接管及 4 条诊断。原始 46 列逐格保留，理论成交链均价、平仓损益隐含均价与单仓位账本隐含标记价分列。此笔记不代表重新抓取或重读原始档案。

In [1]:
from pathlib import Path
import csv, json, hashlib
from collections import Counter
from decimal import Decimal
candidates=[Path.cwd(),Path.cwd()/'results/csv1_full_20250322_20261031']
for candidate in candidates:
    if (candidate/'csv1_20250322_20261031_victim_prices_zh.csv').exists():
        folder=candidate
        output=folder/'csv1_20250322_20261031_victim_prices_zh.csv'
        source=folder/'csv1_20250322_20261031_account_value_zh.csv'
        break
    if (candidate/'csv1_20250322_20261031_含account_value_被接管用户开仓价_清算标记价.csv').exists():
        folder=candidate
        output=folder/'csv1_20250322_20261031_含account_value_被接管用户开仓价_清算标记价.csv'
        source=folder.parent/'csv1_account_value_20250322_20261031_20261109/csv1_20250322_20261031_含account_value.csv'
        break
else:
    raise FileNotFoundError('Run from the repository root or the CSV result folder')
with output.open(encoding='utf-8-sig',newline='') as h: rows=list(csv.reader(h))
with source.open(encoding='utf-8-sig',newline='') as h: original=list(csv.reader(h))
assert len(rows)==len(original)==22049
assert len(rows[0])==58
assert all(r[:46]==s for r,s in zip(rows,original))
print('原始 46 列、行序和诊断行均逐格一致')


原始 46 列、行序和诊断行均逐格一致


In [2]:
header=rows[0]
records=[dict(zip(header,r)) for r in rows[1:]]
fills=[r for r in records if r['记录类型']=='fill']
issues=[r for r in records if r['记录类型']!='fill']
assert len(fills)==22044 and len(issues)==4
assert len({r['接管事件编号'] for r in fills})==22044
assert all(all(v=='' for v in list(r.values())[46:]) for r in issues)
counts={name:sum(bool(r[name]) for r in fills) for name in ['accountValue','被接管用户平均开仓价','平均开仓价_平仓损益隐含值','清算标记价','清算标记价_单仓位账本隐含值']}
print(json.dumps(counts,ensure_ascii=False,indent=2))


{
  "accountValue": 22044,
  "被接管用户平均开仓价": 21878,
  "平均开仓价_平仓损益隐含值": 22041,
  "清算标记价": 22041,
  "清算标记价_单仓位账本隐含值": 3
}


In [3]:
for r in fills:
    for name in ['被接管用户平均开仓价','平均开仓价_平仓损益隐含值','清算标记价','清算标记价_单仓位账本隐含值']:
        if r[name]: assert Decimal(r[name]).is_finite() and Decimal(r[name])>0
    assert r['被接管用户平均开仓价'] or r['平均开仓价未补全原因']
    assert r['清算标记价'] or r['清算标记价未补全原因']
missing_mark=[{k:r[k] for k in ['接管事件编号','被接管用户地址','清算标记价_单仓位账本隐含值','清算标记价未补全原因']} for r in fills if not r['清算标记价']]
assert len(missing_mark)==3
print(json.dumps(missing_mark,ensure_ascii=False,indent=2))


[
  {
    "接管事件编号": "bd5f9795b0ce267c203ad5f3b69495bc7fb7b7923eefd5eb43ea09ef5260053d",
    "被接管用户地址": "0x43205e893e4a958fb993c9a0df436530f47aefb8",
    "清算标记价_单仓位账本隐含值": "13.945",
    "清算标记价未补全原因": "node_trades未记录markPx；官方成交目标窗口未返回该成交；官方txDetails确认清算交易但无markPx；0xArchive Free历史窗口超限403(history_window_exceeded)"
  },
  {
    "接管事件编号": "a9dc9897d8d9f4b9e3bb017af85dcc2280d9707384509719f027a9e7dfef221f",
    "被接管用户地址": "0x880ac484a1743862989a441d6d867238c7aa311c",
    "清算标记价_单仓位账本隐含值": "0.0137380000000000002904514285998015266966491834221669588034397",
    "清算标记价未补全原因": "node_trades未记录markPx；官方成交目标窗口未返回该成交；官方txDetails确认清算交易但无markPx；0xArchive Free历史窗口超限403(history_window_exceeded)"
  },
  {
    "接管事件编号": "b9b21341642d5e4b5d1eb93d9e9fd60b868d1dc5c0e80487b5f4b99c1bac54f3",
    "被接管用户地址": "0x4203c3eb58cf2406acf490221038fdd8b5fcf27e",
    "清算标记价_单仓位账本隐含值": "0.45962",
    "清算标记价未补全原因": "node_trades未记录markPx；官方成交目标窗口未返回该成交；官方txDetails确认清算交易但无markPx；0xArchive Free历史窗口超限403(history_window_exceeded)"


In [4]:
evidence=json.loads((folder/'victim_price_validation.json').read_text(encoding='utf-8'))
assert hashlib.sha256(output.read_bytes()).hexdigest()==evidence['sha256']
assert evidence['counts']['fills']==len(fills)
print('CSV SHA256 与独立回读证据一致')
print(json.dumps(Counter(r['平均开仓价未补全原因'] for r in fills if not r['被接管用户平均开仓价']),ensure_ascii=False,indent=2))


CSV SHA256 与独立回读证据一致
{
  "接管前逐笔startPosition衔接不连续，无法确认完整成本链；最近已读断点 2025-05-30T00:59:31.943+00:00：按上一笔推算仓位 2000000，该笔startPosition 1500000.0，差值 -500000；官方已查询该所需窗口的接口保留范围，仍未取得所需完整成交链": 1,
  "接管前逐笔startPosition衔接不连续，无法确认完整成本链；最近已读断点 2025-06-21T21:25:00.096+00:00：按上一笔推算仓位 0.3778，该笔startPosition 0.4278，差值 0.05；官方已查询该所需窗口的接口保留范围，仍未取得所需完整成交链": 1,
  "接管前逐笔startPosition衔接不连续，无法确认完整成本链；最近已读断点 2025-06-21T21:31:41.128+00:00：按上一笔推算仓位 132.0461，该笔startPosition 1108.1264，差值 976.0803；官方已查询该所需窗口的接口保留范围，仍未取得所需完整成交链": 1,
  "观察到仓位数量衔接，但开仓起点至接管之间缺少连续成交时间覆盖，无法排除遗漏净数量为零的中间交易；尚无连续来源证明的UTC时段：2025-07-19T09:53:22.352+00:00 至 2025-07-19T10:16:52.008+00:00，2025-07-27T08:45:48.665+00:00 至 2025-07-27T08:49:59.369+00:00，2025-07-27T11:53:09.453+00:00 至 2025-07-27T12:01:50.962+00:00；官方已查询该所需窗口的接口保留范围，仍未取得所需完整成交链": 1,
  "已取得平仓记录，但可用历史未追溯至零仓位或反向开仓起点；官方已查询该所需窗口的接口保留范围，仍未取得所需完整成交链": 30,
  "接管前逐笔startPosition衔接不连续，无法确认完整成本链；最近已读断点 2025-08-14T12:35:05.991+00:00：按上一笔推算仓位 0，该笔startPosition 120480.7，差值 120480.7；官方已查询该所需窗口的接口保留范围

均价重建从零仓位或反向开仓余量开始，要求逐笔仓位精确衔接、同毫秒顺序唯一，以及起点到接管时刻的连续可用来源时间覆盖。近期 WebSocket 快照只补充成交，不单独证明历史时间段完整；空接口回执也不能证明更早成交不存在。数量衔接本身不能排除遗漏中间往返交易，可用时间覆盖仍不能保证提供方从未遗漏原始记录。损益隐含值和账本隐含标记价是参考值，不能冒充原生历史 entryPx、entry notional 或原始 markPx。CSV 不包含公式、样式或图表；本笔记以标准 CPython 顺序执行核验，未进行 Jupyter 图形界面渲染。